# Aula 21 — Processamento distribuído de dados
**Apache Spark e PySpark — prática com DataFrames**

**Objetivo:** compreender como o Spark organiza dados em partições e executa transformações e agregações.

Cada ciclo apresenta uma demonstração completa seguida de uma pequena atividade. O desafio final combina as operações utilizadas.

**Contexto teórico:** HDFS armazena dados distribuídos, YARN gerencia recursos e MapReduce organiza o processamento no ecossistema Hadoop. O Spark é um mecanismo de processamento que pode operar independentemente do Hadoop. Nesta prática usaremos a API de DataFrames do PySpark no Databricks, sem SQL nem RDDs.

## Ciclo 0 — Preparação: leituras industriais
A base sintética contém **30 mil leituras** de seis máquinas industriais. Os valores são determinísticos para que os alunos obtenham resultados reproduzíveis.

In [ ]:
from pyspark.sql.functions import (
    col, expr, when, round, spark_partition_id,
    count, avg, sum as sum_
)

df = (
    spark.range(1, 30001)
    .withColumnRenamed("id", "id_leitura")
    .withColumn(
        "maquina_id",
        expr("concat('M', cast((id_leitura % 6) + 1 as string))")
    )
    .withColumn(
        "linha",
        when(col("id_leitura") % 3 == 0, "Linha A")
        .when(col("id_leitura") % 3 == 1, "Linha B")
        .otherwise("Linha C")
    )
    .withColumn(
        "turno",
        when(col("id_leitura") % 2 == 0, "Diurno").otherwise("Noturno")
    )
    .withColumn("temperatura", round(62 + (col("id_leitura") % 40) * 0.8, 1))
    .withColumn("vibracao", round(1.5 + (col("id_leitura") % 13) * 0.7, 1))
    .withColumn("pecas_produzidas", (100 + col("id_leitura") % 60).cast("int"))
    .withColumn("pecas_refugadas", (col("id_leitura") % 8).cast("int"))
)

## Ciclo 1 — DataFrame e inspeção
**Demonstração:** utilizar `printSchema()`, `select()`, `limit()`, `display()` e `count()` para conhecer um DataFrame Spark.

In [ ]:
df.printSchema()
display(df.select("id_leitura", "linha", "temperatura").limit(6))
print("Total de leituras:", df.count())

**Atividade 1 — Faça você:** selecione `maquina_id`, `turno` e `vibracao`. Exiba **6 leituras** e imprima a quantidade total de registros, repetindo o procedimento demonstrado.

In [ ]:
# TODO — Atividade 1
# Selecione as colunas, mostre 6 leituras e conte os registros.

## Ciclo 2 — Transformações e ações
**Demonstração:** `filter()` e `withColumn()` definem transformações; `count()` e `display()` solicitam a execução. Isso ilustra a **avaliação preguiçosa (lazy evaluation)** do Spark.

In [ ]:
dados_tratados = (
    df
    .filter(col("temperatura") >= 80)
    .withColumn(
        "taxa_refugo_pct",
        round(col("pecas_refugadas") / col("pecas_produzidas") * 100, 2)
    )
)

print("Leituras com temperatura >= 80:", dados_tratados.count())
display(
    dados_tratados.select(
        "maquina_id", "temperatura", "taxa_refugo_pct"
    ).limit(6)
)

**Atividade 2 — Faça você:** crie `dados_vibracao`, filtrando a base `df` para `vibracao >= 7`. Acrescente a mesma coluna `taxa_refugo_pct`, conte os registros e exiba **6 leituras**. Use apenas as operações da demonstração.

In [ ]:
# TODO — Atividade 2
# Crie dados_vibracao com filter() e withColumn(); exiba e conte.

## Ciclo 3 — Partições: organização do trabalho
**Demonstração:** `repartition(4)` redistribui os registros em quatro partições lógicas. `spark_partition_id()` identifica a partição de cada registro. **Partição não é sinônimo de nó físico ou executor.**

In [ ]:
dados_particionados = (
    df
    .repartition(4)
    .withColumn("particao", spark_partition_id())
)

display(
    dados_particionados.groupBy("particao")
    .count()
    .orderBy("particao")
)

**Atividade 3 — Faça você:** repita a demonstração com **2 partições**, criando `dados_duas_particoes`. Conte a quantidade de registros em cada partição e exiba o resultado. Compare com as quatro partições.

In [ ]:
# TODO — Atividade 3
# Use repartition(2), spark_partition_id(), groupBy(), count() e orderBy().

## Ciclo 4 — Agregações e plano de execução
**Demonstração:** agrupar as leituras com `groupBy()` e `agg()`, calcular indicadores com `count()`, `avg()` e `round()` e observar o plano com `explain()`.

In [ ]:
indicadores_linhas = (
    dados_tratados
    .groupBy("linha")
    .agg(
        count("*").alias("total_leituras"),
        round(avg("temperatura"), 2).alias("temperatura_media"),
        round(avg("taxa_refugo_pct"), 2).alias("taxa_refugo_media_pct")
    )
    .orderBy(col("taxa_refugo_media_pct").desc())
)

display(indicadores_linhas)
indicadores_linhas.explain(mode="formatted")

**Atividade 4 — Faça você:** usando `dados_tratados`, crie `indicadores_maquinas`, agrupando por `maquina_id`. Calcule **total de leituras**, **temperatura média** e **taxa média de refugo**, como na demonstração. Ordene pela maior taxa de refugo e exiba os resultados.

In [ ]:
# TODO — Atividade 4
# Agrupe por maquina_id e repita as três métricas da demonstração.

## Desafio final — Ranking de máquinas sob condições críticas
A equipe de manutenção quer identificar as máquinas com maior refugo **nas leituras com temperatura a partir de 85 °C**.

Partindo de `df`:
1. filtre os registros com `temperatura >= 85`;
2. crie `taxa_refugo_pct` usando a mesma fórmula do Ciclo 2;
3. use `repartition(3)` para distribuir os dados em três partições;
4. agrupe por `maquina_id`, calculando `total_leituras`, `temperatura_media` e `taxa_refugo_media_pct`;
5. ordene pela maior `taxa_refugo_media_pct`, exiba o ranking e consulte o plano com `explain(mode="formatted")`.

**Interpretação:** qual máquina ficou em primeiro lugar? Onde o plano indica movimentação ou agrupamento de dados entre partições?

**Utilize apenas operações demonstradas durante a aula.**

In [ ]:
# TODO — Desafio final
# Gere ranking_maquinas, exiba o resultado e consulte seu plano de execução.